# Analyze Unmatched Facts from WikiGap Data

This notebook reads annotation files and extracts facts that are not present in both language versions.

In [5]:
# Step 1: Read a file from scratch/annotation_save/wikigap_data/ and print its content

import json
from pathlib import Path
import pprint

# Define the path to the data directory
data_dir = Path("scratch/annotation_save/wikigap_data")

# Choose a file (you can change this to any other file in the directory)
file_path = data_dir / "annotation_2025-11-07_Armand Sabatier_fr.json"

# Read and parse the JSON file
with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# Print the content
print(f"File: {file_path.name}")
print(f"\nData structure keys: {list(data.keys())}")
print(f"\nNumber of columns: {len(data['columns'])}")
print(f"\nColumn names:")
for col in data['columns']:
    print(f"  - {col['name']}: {len(col['values'])} values")

# Display first few facts
facts_col = next(col for col in data['columns'] if col['name'] == 'fact')
print(f"\nFirst 5 facts:")
for i, fact in enumerate(facts_col['values'][:5]):
    print(f"{i+1}. {fact}")

# Display an example value from each column
print("\n" + "=" * 80)
print("=== EXAMPLE VALUES FROM EACH COLUMN ===\n")

# Extract columns dictionary
columns = {col['name']: col['values'] for col in data['columns']}

# Get an example index (let's use index 1 which has intersection_label = 'no')
example_idx = 1

for col_name, values in columns.items():
    print(f"Column: {col_name}")
    print(f"Type: {type(values[example_idx])}")
    print(f"Example value (index {example_idx}):")
    
    # Pretty print for better readability
    if isinstance(values[example_idx], (list, dict)):
        pprint.pprint(values[example_idx], indent=2, width=100)
    else:
        print(f"  {values[example_idx]}")
    
    print("\n" + "-" * 80 + "\n")

File: annotation_2025-11-07_Armand Sabatier_fr.json

Data structure keys: ['columns']

Number of columns: 11

Column names:
  - fact: 44 values
  - fact_index: 44 values
  - person_name: 44 values
  - fact_aligned_sentence: 44 values
  - src_context: 44 values
  - tgt_contexts: 44 values
  - tgt_fact_indices: 44 values
  - tgt_fact_aligned_sentences: 44 values
  - paragraph_index: 44 values
  - intersection_label: 44 values
  - language: 44 values

First 5 facts:
1. Charles Paul Dieudonné Armand Sabatier est un zoologiste français.
2. Il fait ses études à Montpellier, où il suit les cours de mathématiques spéciales au lycée, puis s'inscrit en médecine.
3. Il donne ces cours à la faculté de théologie protestante de Montauban.
4. Il épouse Laure Gervais de Rouville, ils ont une fille, Jeanne.
5. Charles Paul Dieudonné Armand Sabatier est né à Ganges.

=== EXAMPLE VALUES FROM EACH COLUMN ===

Column: fact
Type: <class 'str'>
Example value (index 1):
  Il fait ses études à Montpellier, où 

In [7]:
# Step 2: Store unmatched facts in a dictionary (language-agnostic)

def extract_unmatched_facts(data):
    """
    Extract facts that are not matched between languages.
    A fact is unmatched if intersection_label is 'no'.
    Returns a dictionary with language codes as keys and lists of unmatched facts as values.
    """
    # Find the relevant columns
    columns = {col['name']: col['values'] for col in data['columns']}
    
    facts = columns['fact']
    languages = columns['language']
    intersection_labels = columns['intersection_label']
    
    # Initialize the result dictionary dynamically based on languages present
    unmatched_facts = {}
    
    # Iterate through all facts
    for fact, lang, label in zip(facts, languages, intersection_labels):
        # Initialize language key if not present
        if lang not in unmatched_facts:
            unmatched_facts[lang] = []
        
        # Check if the fact is unmatched (intersection_label == 'no')
        if label == 'no':
            unmatched_facts[lang].append(fact)
    
    return unmatched_facts

# Extract unmatched facts
unmatched = extract_unmatched_facts(data)

# Print results
print(f"\n=== UNMATCHED FACTS ===")
print(f"\nLanguages found: {list(unmatched.keys())}")

for lang, facts in unmatched.items():
    print(f"\n--- {lang.upper()} Unmatched Facts: {len(facts)} ---")
    for i, fact in enumerate(facts, 1):
        print(f"{i}. {fact}")

# Store in dictionary as requested
unmatched_facts_dict = unmatched
print(f"\n=== Dictionary stored as 'unmatched_facts_dict' ===")
print(f"Languages: {list(unmatched_facts_dict.keys())}")
for lang, facts in unmatched_facts_dict.items():
    print(f"{lang.upper()}: {len(facts)} facts")


=== UNMATCHED FACTS ===

Languages found: ['fr', 'en']

--- FR Unmatched Facts: 26 ---
1. Il fait ses études à Montpellier, où il suit les cours de mathématiques spéciales au lycée, puis s'inscrit en médecine.
2. Il donne ces cours à la faculté de théologie protestante de Montauban.
3. Il épouse Laure Gervais de Rouville, ils ont une fille, Jeanne.
4. Charles Paul Dieudonné Armand Sabatier est doyen de la faculté des sciences.
5. Édouard Marsal est peintre.
6. Il se montre très favorable à la théorie de l'évolutionnisme.
7. Il dirige la station de zoologie maritime de Sète.
8. Il est doyen de la faculté des sciences de 1891 à 1904.
9. Charles Paul Dieudonné Armand Sabatier est un médecin français.
10. Charles Paul Dieudonné Armand Sabatier est mort à Montpellier.
11. Il donne une série de cours sur l'évolutionnisme.
12. Ce portrait est déposé à la faculté des sciences montpelliéraine.
13. Il est enterré au cimetière protestant de Montpellier.
14. Il est membre de l'Académie des scienc

In [10]:
# Step 3: Evaluate cultural importance of unmatched facts using OpenAI

import os
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("THE_KEY"))

# Language to country/culture mapping
LANG_COUNTRY_MAPPING = {
    'en': 'English-speaking countries (US, UK, etc.)',
    'fr': 'France',
    'it': 'Italy',
    'de': 'Germany',
    'es': 'Spain',
    'ru': 'Russia',
    'zh': 'China',
    'ja': 'Japan',
    'ar': 'Arab countries',
    'pt': 'Portugal/Brazil',
    'nl': 'Netherlands',
    'pl': 'Poland',
    'ko': 'Korea',
}

def evaluate_cultural_importance(fact, language, person_name):
    """
    Use OpenAI to evaluate if a fact is culturally important for the country/culture
    associated with the given language.
    """
    country = LANG_COUNTRY_MAPPING.get(language, f"the {language}-speaking region")
    
    prompt = f"""You are evaluating whether a specific fact about a person is culturally important or relevant to {country}.

Person: {person_name}
Fact: {fact}

Question: Is this fact particularly important or relevant to the culture, history, or context of {country}?

Please respond with:
1. A rating: HIGH, MEDIUM, or LOW
2. A brief explanation (1-2 sentences)

Format your response as:
RATING: [HIGH/MEDIUM/LOW]
EXPLANATION: [your explanation]"""

    try:
        response = client.chat.completions.create(
            model="gpt-5-mini",
            messages=[
                {"role": "system", "content": "You are an expert in cultural studies and international relations, skilled at evaluating the cultural significance of biographical facts."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.3,
            max_tokens=200
        )
        
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"ERROR: {str(e)}"

# Get person name from the data
columns = {col['name']: col['values'] for col in data['columns']}
person_name = columns['person_name'][0]  # Assuming same person throughout

print(f"=== EVALUATING CULTURAL IMPORTANCE ===")
print(f"Person: {person_name}\n")

# Store results
cultural_evaluations = {}

for lang, facts in unmatched_facts_dict.items():
    print(f"\n{'='*80}")
    print(f"Language: {lang.upper()} - Culture: {LANG_COUNTRY_MAPPING.get(lang, 'Unknown')}")
    print(f"{'='*80}\n")
    
    cultural_evaluations[lang] = []
    
    for i, fact in enumerate(facts, 1):
        print(f"Fact {i}: {fact}")
        print("\nEvaluating...")
        
        evaluation = evaluate_cultural_importance(fact, lang, person_name)
        cultural_evaluations[lang].append({
            'fact': fact,
            'evaluation': evaluation
        })
        
        print(evaluation)
        print("\n" + "-"*80 + "\n")

print(f"\n=== EVALUATION COMPLETE ===")
print(f"Results stored in 'cultural_evaluations' dictionary")

=== EVALUATING CULTURAL IMPORTANCE ===
Person: Armand Sabatier


Language: FR - Culture: France

Fact 1: Il fait ses études à Montpellier, où il suit les cours de mathématiques spéciales au lycée, puis s'inscrit en médecine.

Evaluating...
RATING: MEDIUM  
EXPLANATION: Armand Sabatier's education in Montpellier, a city known for its historical significance in medicine and academia, reflects the broader cultural value placed on education and intellectual pursuit in France. However, without additional context about his contributions or influence, this fact alone does not carry high cultural weight.

--------------------------------------------------------------------------------

Fact 2: Il donne ces cours à la faculté de théologie protestante de Montauban.

Evaluating...
RATING: MEDIUM  
EXPLANATION: Armand Sabatier's education in Montpellier, a city known for its historical significance in medicine and academia, reflects the broader cultural value placed on education and intellectual p

In [11]:
# Step 4: Summarize cultural importance ratings

import re
from collections import Counter

def extract_rating(evaluation_text):
    """Extract rating from evaluation text."""
    match = re.search(r'RATING:\s*(HIGH|MEDIUM|LOW)', evaluation_text, re.IGNORECASE)
    if match:
        return match.group(1).upper()
    return 'UNKNOWN'

print("="*80)
print("=== CULTURAL IMPORTANCE SUMMARY ===")
print("="*80)

# Analyze ratings for each language
for lang, evaluations in cultural_evaluations.items():
    print(f"\n{lang.upper()} - {LANG_COUNTRY_MAPPING.get(lang, 'Unknown')}")
    print("-" * 80)
    
    # Extract all ratings
    ratings = [extract_rating(item['evaluation']) for item in evaluations]
    rating_counts = Counter(ratings)
    
    total = len(ratings)
    print(f"Total unmatched facts: {total}")
    print(f"\nRating distribution:")
    for rating in ['HIGH', 'MEDIUM', 'LOW', 'UNKNOWN']:
        count = rating_counts.get(rating, 0)
        percentage = (count / total * 100) if total > 0 else 0
        print(f"  {rating}: {count} ({percentage:.1f}%)")
    
    # Show HIGH importance facts
    high_importance_facts = [
        item['fact'] for item in evaluations 
        if extract_rating(item['evaluation']) == 'HIGH'
    ]
    
    if high_importance_facts:
        print(f"\nHIGH importance facts ({len(high_importance_facts)}):")
        for i, fact in enumerate(high_importance_facts, 1):
            print(f"  {i}. {fact[:100]}{'...' if len(fact) > 100 else ''}")
    else:
        print(f"\nNo HIGH importance facts found.")
    
    print()

# Overall summary
print("\n" + "="*80)
print("=== OVERALL SUMMARY ===")
print("="*80)

all_ratings = []
for lang, evaluations in cultural_evaluations.items():
    all_ratings.extend([extract_rating(item['evaluation']) for item in evaluations])

overall_counts = Counter(all_ratings)
total_facts = len(all_ratings)

print(f"Total unmatched facts across all languages: {total_facts}")
print(f"\nOverall rating distribution:")
for rating in ['HIGH', 'MEDIUM', 'LOW', 'UNKNOWN']:
    count = overall_counts.get(rating, 0)
    percentage = (count / total_facts * 100) if total_facts > 0 else 0
    print(f"  {rating}: {count} ({percentage:.1f}%)")

print("\n" + "="*80)

=== CULTURAL IMPORTANCE SUMMARY ===

FR - France
--------------------------------------------------------------------------------
Total unmatched facts: 26

Rating distribution:
  HIGH: 4 (15.4%)
  MEDIUM: 16 (61.5%)
  LOW: 6 (23.1%)
  UNKNOWN: 0 (0.0%)

HIGH importance facts (4):
  1. Ce buste est inscrit sur la liste d'objets des Monuments historiques.
  2. Ce portrait est inscrit sur la liste des objets des Monuments historiques.
  3. Durant la guerre franco-allemande de 1870, il est chirurgien responsable des ambulances du midi.
  4. Il est membre correspondant de l'Académie des sciences (1895-1910).


EN - English-speaking countries (US, UK, etc.)
--------------------------------------------------------------------------------
Total unmatched facts: 2

Rating distribution:
  HIGH: 0 (0.0%)
  MEDIUM: 0 (0.0%)
  LOW: 2 (100.0%)
  UNKNOWN: 0 (0.0%)

No HIGH importance facts found.


=== OVERALL SUMMARY ===
Total unmatched facts across all languages: 28

Overall rating distribution:
 